In [ ]:
# Install Cadabra if necessary, and enable Cadabra input in this notebook.
try:
    import cadabra2
except ImportError:
    %pip install -q cadabra2
%load_ext cadabra2_jupyter

# Introduction to Cadabra for beginners

This notebook introduces Cadabra to people who have never used it before, or who have
used one of the old 1.x versions. It will show you how to enter expressions, attach
properties to patterns, perform simple substitutions and so on. 

If you need help, do not hesitate to email [help@cadabra.science](mailto:help@cadabra.science) or visit
the web site at <http://cadabra.science/>.


# Equations of motion

A good example to illustrate a number of Cadabra manipulations is to derive the equations of
motion of a field theory from an action. Below we discuss the Maxwell equations, just for
simplicity.

In Cadabra, the default setting is to do very little automatic simplification (essentially, only
equal terms are collected). You can change this, however, by writing a `post_process`
function which contains all the algorithms which you want to run at every step of your notebook.
Below is an example:

In [ ]:
def post_process(ex):
	sort_product(ex)
	canonicalise(ex)
	collect_terms(ex)

As you can see, this sorts factors in a product (alphabetically, you can change this if you want), then 
writes indices in canonical form, and collects equal terms.

Before we can enter our Maxwell action, we need to declare some properties of the symbols which we
want to use. Cadabra is fairly minimalistic with this; you typically only have to declare what
you are going to use. Below we declare that $\mu$, $\nu$, $\rho$ are indices, $x$ is a coordinate
label, and `\partial` indicates a partial derivative. We then declare properties of the field 
strength and the gauge potential. This should all read fairly naturally.

In [ ]:
{\mu,\nu,\rho}::Indices(position=free).
x::Coordinate.
\partial{#}::Derivative.

F_{\mu\nu}::AntiSymmetric;
F_{\mu\nu}::Depends(x).
A_{\mu}::Depends(x,\partial{#}).
\delta{#}::Accent;

Note how lines ending in a semi-colon get their output printed, while those ending in a period do not.

We are now ready to write down our action, very similar to what you would write down in LaTeX. We also 
write down a rule which says how the field strength is related to the gauge potential. Note how this rule
does not automatically get inserted into the action; you need to tell Cadabra to do that.

In [ ]:
S:= -1/4 \int{ F_{\mu\nu} F^{\mu\nu} }{x};

In [ ]:
rl:= F_{\mu\nu} = \partial_{\mu}{A_{\nu}} - \partial_{\nu}{A_{\mu}};

In [ ]:
substitute(S, rl);

Note how this has automatically replaced the field strength with upper as well as
lower indices, by virtue of the fact that the index position was declared with `position=free`.

We can now compute a variational derivative of this object with respect to $A_\mu$.
Note the use of inline formulas using the standard TeX \$...\$ construction.

In [ ]:
vary(S, $A_{\mu} -> \delta{A_{\mu}}$);

We need to distribute the products over the sums, and then integrate by parts, away from
the $\delta{A_{\mu}}$ factors:

In [ ]:
distribute(S);
integrate_by_parts(S, $\delta{A_{\mu}}$);

Finally, in order to write this back in terms of the field strength, we use the following rewriting trick,

In [ ]:
substitute(_, $\partial_{\mu}{A_{\nu}} -> 1/2 \partial_{\mu}{A_{\nu}} + 1/2 F_{\mu\nu} + 1/2 \partial_{\nu}{A_{\mu}}$);

In [ ]:
distribute(_);

This isolates the equations of motion $\partial^\nu F_{\mu\nu}=0$ in the integrand.